# exp-07 — three models on the full checklist

Reads the runs under `experiments/runs/exp-07-models/`, scores every leaf not
scored yet, and compares **Opus 5.5, Sonnet 5.5 and Haiku 5.5** on exp-06's
baseline configuration — `fig-checklist-exp07`, check-major
`A|B <- C_i <- D`, one session per figure.

**This notebook spends nothing.** The sessions are produced by

```bash
python experiments/exp-07-models/run.py
```

The endpoints, the ±0.02 margin and the extension rule are fixed in
[`thinking/experiments/exp-07-models.md`](../../thinking/experiments/exp-07-models.md).
The rule says whether to extend to five replicates; the author decides.

## The gold

**`gold-v6`**. This cell pins the scorer to it and must run first.

In [ ]:
from soda_mmqc.gold import pin_gold

GOLD_TAG = "gold-v6"
pin_gold(GOLD_TAG)

In [ ]:
import importlib.util
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from soda_mmqc.config import gold_root
from soda_mmqc.core.run_layout import iter_leaves
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check, stale_examples
from soda_mmqc.reporting import (
    layer1_counts_by_example,
    layer2_counts,
    layer_s_counts_by_example,
    load_run_root,
    non_response_counts,
    plot_stacked_counts,
    scores_frame,
)
from soda_mmqc.reporting.styles import (
    LAYER1_ORDER,
    LAYER2_BINARY_ORDER,
    LAYER2_GRADED_ORDER,
    LAYER_S_ORDER,
)
from soda_mmqc.reporting.tables import LAYER1_OUTLIER_OUTCOMES, LAYER2_ERROR_OUTCOMES

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)


def _load(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


exp07 = _load("exp07_run", Path("../../experiments/exp-07-models/run.py"))
panels = _load("exp04_panels", Path("../../experiments/exp-04-panel-major/panels.py"))
assert exp07.GOLD_TAG == GOLD_TAG, "the runner and this notebook name different gold"

CHECKLIST, ENTRY, CHECKS, CLASSIFY = exp07.CHECKLIST, exp07.ENTRY, exp07.CHECKS, exp07.CLASSIFY
MODELS = exp07.MODELS
SHORT = {"claude-opus-5-5": "Opus 5.5", "claude-sonnet-5-5": "Sonnet 5.5", "claude-haiku-5-5": "Haiku 5.5"}
#: The CLI's own background model, in every session's model usage.
BACKGROUND = "claude-haiku-5-5"
PAIRS = [("claude-opus-5-5", "claude-sonnet-5-5"), ("claude-sonnet-5-5", "claude-haiku-5-5"),
         ("claude-opus-5-5", "claude-haiku-5-5")]
MARGIN = 0.02

#: One decision-like property per check: its decision, or what plays that part.
DECISIONS = {
    "micrograph-scale-bar": "scale_bar_defined_in_caption",
    "image-annotation-defined": "annotation_defined_in_caption",
    "single-channel-for-overlay": "all_channels_shown_separately",
    "individual-data-points": "decision",
    "error-bars-defined": "decision",
    "plot-axis-units": "decision",
    "plot-gap-labeling": "decision",
    "stat-significance-level": "decision",
    "stat-test": "decision",
    "replication-reporting": "decision",
    "panel-image-matches-caption": "image_matches_caption",
}
assert set(DECISIONS) == set(CHECKS)

#: The full run; set EXP07_ROOT to `.../smoke` to exercise the notebook.
ROOT = Path(os.environ.get("EXP07_ROOT", "../../experiments/runs/exp-07-models"))
GOLD = gold_root()
SEED = 20261012
RESAMPLES = 10_000


def model_leaves():
    """(model, replicate, leaf) for every model that has run."""
    for model in MODELS:
        if (ROOT / model).is_dir():
            for _, replicate, leaf in iter_leaves(ROOT / model):
                yield model, replicate, leaf


def owning_check(value: str) -> str | None:
    for check in CHECKS:
        if value == check or value.startswith((f"{check}[", f"{check}.")):
            return check
    return None


def to_per_check(frame: pd.DataFrame, model: str) -> pd.DataFrame:
    """`<check>[].x` -> (`check`, `outputs[].x`); list key `<check>` -> `outputs`; + `condition`."""
    frame = frame.copy()
    frame["condition"] = model
    key = "property" if "property" in frame else "list_key"
    frame["check"] = [owning_check(v) for v in frame[key]]
    nested = frame["check"].isna()
    if key == "list_key" and nested.any():
        # plot-axis-units' per-axis lists keep their own keys.
        frame.loc[nested, "check"] = "plot-axis-units"
    if frame["check"].isna().any():
        raise ValueError(f"{key}s owned by no check: "
                         f"{sorted(frame.loc[frame['check'].isna(), key].unique())}")
    for column in ("property", "list_key"):
        if column in frame:
            frame[column] = ["outputs" + v[len(c):] if v.startswith(c) else v
                             for v, c in zip(frame[column], frame["check"])]
    return frame


def is_decision(check: str, prop: str) -> bool:
    return DECISIONS.get(check) == prop.removeprefix("outputs[].")


def bootstrap_mean(values, rng) -> tuple[float, float, float, int]:
    v = np.asarray([x for x in values if not pd.isna(x)], dtype=float)
    if not len(v):
        return (np.nan, np.nan, np.nan, 0)
    boots = rng.choice(v, size=(RESAMPLES, len(v)), replace=True).mean(axis=1)
    return (v.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5), len(v))


print(ROOT.resolve(), "models run:", [m for m in MODELS if (ROOT / m).is_dir()])

## Score every leaf that is not scored yet

In [ ]:
# Compared by content (core.scoring.stale_examples), not by file time:
# a branch checkout rewrites predictions, and extracting a pinned gold
# snapshot rewrites gold, with new mtimes and the same bytes.
stale = {}
for model, replicate, leaf in model_leaves():
    if not (leaf / ANALYSIS_FILENAME).is_file():
        score_check(CHECKLIST, ENTRY, leaf, save=True)
        print(f"scored {model} rep-{replicate:02d}")
        continue
    changed = stale_examples(CHECKLIST, ENTRY, leaf)
    if changed:
        stale[leaf] = changed

if stale:
    raise RuntimeError(
        "predictions or gold differ from what their analysis scored in "
        f"{len(stale)} leaf/leaves; delete those analysis.json and re-run "
        "this cell:\n  "
        + "\n  ".join(f"{leaf} ({len(examples)} example(s))" for leaf, examples in stale.items())
    )
print("all leaves scored, none stale")

## 0. Dispatch, read first

Per session, from `skill_trace.json`: how many of the eleven checks were
invoked and which were not; how often `classify-panels` was invoked; whether
the checks ran in D's order; any skill outside the closure. A session that
invoked some checks but not all is **partial**; exp-06 saw none, and the
smoke tests saw it in Sonnet 5.5 (stopping after the first check) and Haiku
5.5 (skipping `single-channel-for-overlay`).

In [ ]:
def read_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


rows = []
for model, replicate, leaf in model_leaves():
    for trace in leaf.rglob("intermediates/skill_trace.json"):
        skills = [e.get("skill") for e in read_json(trace) or [] if isinstance(e, dict) and e.get("skill")]
        called = list(dict.fromkeys(s for s in skills if s in CHECKS))
        rows.append({
            "condition": model, "replicate": replicate,
            "example": str(trace.parent.parent.relative_to(leaf)),
            "n_checks": len(called),
            **{c: c in called for c in CHECKS},
            "classify_calls": skills.count(CLASSIFY),
            "in_order": called == [c for c in CHECKS if c in called],
            "spontaneous": tuple(sorted(set(skills) - {ENTRY, CLASSIFY, *CHECKS})),
        })
sessions = pd.DataFrame(rows)
sessions["dispatch"] = np.select([sessions["n_checks"] == len(CHECKS), sessions["n_checks"] == 0],
                                 ["all eleven", "no check"], "partial")
print(sessions.groupby(["condition", "dispatch"]).size().unstack(fill_value=0).reindex(MODELS))
print()
print("checks invoked in partial sessions:")
print(sessions[sessions["dispatch"] == "partial"].groupby(["condition", "n_checks"]).size())

In [ ]:
# Per check, in D's order: the share of sessions that invoked it.
sessions.groupby("condition")[list(CHECKS)].mean().T[[m for m in MODELS if m in set(sessions["condition"])]]

In [ ]:
print("classify-panels calls per session:")
print(sessions.groupby(["condition", "classify_calls"]).size().unstack(fill_value=0))
print("in D's order:", sessions.groupby("condition")["in_order"].agg(["sum", "count"]).to_dict("index"))
spont = sessions.explode("spontaneous").dropna(subset=["spontaneous"])
print("spontaneous skills:", "none" if spont.empty else spont.groupby(["condition", "spontaneous"]).size().to_dict())

## The model that answered: fallback and refusals

From each session's `tool_audit.json`: the models in its usage and, per
assistant message, the model that wrote it. Claude Code re-runs a request its
safety classifier flags on a fallback model and continues there (Opus 5.5:
biology to Opus 5, cybersecurity to Opus 4.8; Sonnet 5.5: cybersecurity to
Sonnet 5, biology a refusal). Allowed and recorded, as decided. The CLI's
background model, `claude-haiku-5-5`, is set apart — and cannot be told apart
from the Haiku arm's own messages by name.

In [ ]:
rows = []
for model, replicate, leaf in model_leaves():
    for audit_path in leaf.rglob("intermediates/tool_audit.json"):
        audit = read_json(audit_path) or {}
        usage = audit.get("usage") or {}
        per_model = usage.get("model_usage") or {}
        others = sorted(m for m in per_model if m not in (model, BACKGROUND))
        sequence = []
        for turn in audit.get("turns") or []:
            if not sequence or sequence[-1] != turn["model"]:
                sequence.append(turn["model"])
        tokens = usage.get("usage") or {}
        rows.append({
            "condition": model, "replicate": replicate,
            "example": str(audit_path.parent.parent.relative_to(leaf)),
            "other_models": tuple(others),
            "other_output_tokens": sum((per_model[m] or {}).get("outputTokens") or 0 for m in others),
            "requested_output_tokens": (per_model.get(model) or {}).get("outputTokens") or 0,
            "sequence": " -> ".join(sequence),
            "stop_reason": usage.get("stop_reason"),
            "terminal_reason": usage.get("terminal_reason"),
            "cost": usage.get("total_cost_usd"),
            "background_cost": (per_model.get(BACKGROUND) or {}).get("costUSD") if model != BACKGROUND else None,
            "turns": usage.get("num_turns"),
            "seconds": (usage.get("duration_ms") or 0) / 1000,
            "input_tokens": sum(tokens.get(k) or 0 for k in
                                ("input_tokens", "cache_creation_input_tokens", "cache_read_input_tokens")),
            "output_tokens": tokens.get("output_tokens") or 0,
            "concurrency": (audit.get("session") or {}).get("concurrency"),
        })
usage = pd.DataFrame(rows)
usage["fallback"] = usage["other_models"].map(bool)
print("sessions with another model:")
print(usage.groupby("condition")["fallback"].agg(["sum", "count"]).reindex(MODELS))
print()
print("to which:", usage[usage["fallback"]].groupby(["condition", "other_models"]).size().to_dict() or "none")
print("stop reasons:", usage.groupby(["condition", "stop_reason"], dropna=False).size().to_dict())
usage[usage["fallback"]][["condition", "replicate", "example", "other_models", "requested_output_tokens",
                          "other_output_tokens", "sequence"]]

### The examples that triggered a fallback

Per figure: which model switched, to which, in how many of its replicates.
A biology flag follows the figure's subject — `s44318-026-00715-1`, for
instance, is about mycobacteria (*M. tuberculosis*, *M. smegmatis*), which are
pathogens — so the same figure can switch again in another replicate, and not
every time.

In [ ]:
fallback_by_example = (
    usage.assign(switched=usage["fallback"])
    .groupby(["example", "condition"])
    .agg(replicates_switched=("switched", "sum"), replicates_run=("switched", "size"),
         fallback_models=("other_models", lambda s: sorted({m for t in s for m in t})),
         output_tokens_on_fallback=("other_output_tokens", "sum"))
    .reset_index()
)
fallback_by_example = fallback_by_example[fallback_by_example["replicates_switched"] > 0]
fallback_by_example if len(fallback_by_example) else "no example triggered a fallback"

## One row per property, per replicate, per example

In [ ]:
frames = {"scores": [], "layer_s": [], "layer_1": [], "layer_2": [], "non_response": []}
for model in MODELS:
    if not (ROOT / model).is_dir():
        continue
    runs = load_run_root(ROOT / model, checklist=CHECKLIST, check=ENTRY)
    frames["scores"].append(to_per_check(scores_frame(runs), model))
    frames["layer_s"].append(to_per_check(layer_s_counts_by_example(runs), model))
    frames["layer_1"].append(to_per_check(layer1_counts_by_example(runs), model))
    frames["layer_2"].append(to_per_check(layer2_counts(runs), model))
    frames["non_response"].append(to_per_check(non_response_counts(runs), model))
scores, layer_s, layer_1, layer_2, non_response = (pd.concat(frames[k], ignore_index=True) for k in frames)
PRESENT = [m for m in MODELS if m in set(scores["condition"])]
print(f"{len(scores)} score rows; replicates per model:",
      scores.groupby("condition")["replicate"].nunique().to_dict())

## Failures and empty answers

A failed session loses all eleven checks.

In [ ]:
print(non_response[non_response["list_key"] == "outputs"]
      .groupby(["check", "condition"])["empty"].sum().unstack(fill_value=0))
wanted = json.loads((Path("../../soda_mmqc/data/checklist") / CHECKLIST / ENTRY / "benchmark.json")
                    .read_text(encoding="utf-8"))["examples"]
missing = [{"condition": model, "replicate": rep,
            "without_prediction": sum(not (leaf / ex / "prediction.json").is_file() for ex in wanted)}
           for model, rep, leaf in model_leaves()]
pd.DataFrame(missing).groupby("condition")["without_prediction"].sum()

## The outcomes, model beside model

In [ ]:
plot_stacked_counts(
    layer_s[layer_s['list_key'] == 'outputs'], group="check", category="outcome", series="condition",
    order=LAYER_S_ORDER, series_order=PRESENT, ylabel="rows",
    title="Layer S row outcomes, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_1, group="check", category="layer1", series="condition",
    order=LAYER1_ORDER, series_order=PRESENT, ylabel="instances",
    title="Layer 1 applicability calls, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2['layer2'].isin(LAYER2_BINARY_ORDER)], group="check", category="layer2", series="condition",
    order=LAYER2_BINARY_ORDER, series_order=PRESENT, ylabel="instances",
    title="Layer 2, binary properties, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2['layer2'].isin(LAYER2_GRADED_ORDER)], group="check", category="layer2", series="condition",
    order=LAYER2_GRADED_ORDER, series_order=PRESENT, ylabel="instances",
    title="Layer 2, graded and multiclass properties, summed over examples and replicates",
)

## Layer S per check, and panels across the eleven lists

Per check, `correct / (correct + missing)` and
`correct / (correct + missing + spurious)`: per-figure rates, replicates
pooled, mean over figures with a bootstrap interval. Then, per figure, the
panels stated across all eleven check lists against the gold's.

In [ ]:
def per_example_rate(frame, category, numerator, denominator=None, by=("check",)):
    counts = (frame.groupby(["condition", *by, "example", category])["count"].sum()
              .unstack(fill_value=0))
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den = counts[[c for c in (denominator or counts.columns) if c in counts]].sum(axis=1)
    return (num / den.where(den > 0)).rename("rate").reset_index()


rng = np.random.default_rng(SEED)
rows = []
rows_s = layer_s[layer_s["list_key"] == "outputs"]
for label, den in (("found", ("correct_row", "missing_row")),
                   ("both errors", ("correct_row", "missing_row", "spurious_row"))):
    r = per_example_rate(rows_s, "outcome", ("correct_row",), den)
    for (cond, check), sub in r.groupby(["condition", "check"]):
        mean, low, high, n = bootstrap_mean(sub["rate"], rng)
        rows.append({"rate": label, "condition": cond, "check": check, "mean": mean,
                     "ci_low": low, "ci_high": high, "n_examples": n})
layer_s_rates = pd.DataFrame(rows)
layer_s_rates.pivot_table(index="check", columns=["rate", "condition"], values="mean").reindex(CHECKS)

In [ ]:
rows = []
for model, replicate, leaf in model_leaves():
    for prediction in leaf.rglob("prediction.json"):
        example = str(prediction.parent.relative_to(leaf))
        gold = json.loads((GOLD / example / "checks" / ENTRY / "expected_output.json")
                          .read_text(encoding="utf-8"))
        labels = [r["panel_label"] for r in gold[CHECKS[0]]]
        rows.append({"condition": model, "replicate": replicate, "example": example,
                     **panels.panel_counts(labels, json.loads(prediction.read_text()), "cm", CHECKS)})
panel_rows = pd.DataFrame(rows)
totals = panel_rows.groupby("condition")[["spurious_panels", "missing_panels",
                                         "partly_missing_panels", "gold_panels"]].sum()
totals["sessions"] = panel_rows.groupby("condition").size()
totals.reindex(PRESENT).fillna(0)

## Layer 1, per check

In [ ]:
rng = np.random.default_rng(SEED)
r = per_example_rate(layer_1, "layer1", ("correct_applicable", "correct_NA"))
rows = []
for (cond, check), sub in r.groupby(["condition", "check"]):
    mean, low, high, n = bootstrap_mean(sub["rate"], rng)
    rows.append({"condition": cond, "check": check, "mean": mean, "ci_low": low,
                 "ci_high": high, "n_examples": n})
layer_1_rates = pd.DataFrame(rows)
layer_1_rates.pivot_table(index="check", columns="condition", values="mean").reindex(CHECKS)[PRESENT]

## Layer 2, per check and per property

`mean_score` on applicable instances: per figure, averaged over replicates;
then the mean over figures with a bootstrap interval.

In [ ]:
rng = np.random.default_rng(SEED)
per_fig = (scores.dropna(subset=["mean_score"])
           .groupby(["condition", "check", "property", "example"])["mean_score"].mean().reset_index())
rows = []
for (cond, check, prop), sub in per_fig.groupby(["condition", "check", "property"]):
    mean, low, high, n = bootstrap_mean(sub["mean_score"], rng)
    rows.append({"condition": cond, "check": check, "property": prop, "mean": mean,
                 "ci_low": low, "ci_high": high, "n_examples": n})
layer_2_props = pd.DataFrame(rows)
layer_2_props["check"] = pd.Categorical(layer_2_props["check"], CHECKS, ordered=True)
layer_2_props.pivot_table(index=["check", "property"], columns="condition",
                          values="mean", observed=True)[PRESENT]

## Layer 2, each check's decision

One decision-like property per check, read on its own rather than averaged
with the check's other properties.

In [ ]:
decisions = layer_2_props[[is_decision(c, p) for c, p in
                           zip(layer_2_props["check"], layer_2_props["property"])]].copy()
decisions["check"] = decisions["check"].astype(str)
decisions.pivot_table(index="check", columns="condition",
                      values=["mean", "ci_low", "ci_high", "n_examples"]).reindex(CHECKS).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.6 * len(CHECKS) + 1.5))
height = 0.8 / max(len(PRESENT), 1)
for i, cond in enumerate(PRESENT):
    sub = decisions[decisions["condition"] == cond].set_index("check").reindex(CHECKS)
    y = np.arange(len(CHECKS)) + (i - (len(PRESENT) - 1) / 2) * height
    ax.barh(y, sub["mean"], height=height, label=SHORT[cond],
            xerr=[sub["mean"] - sub["ci_low"], sub["ci_high"] - sub["mean"]], capsize=2)
ax.set_yticks(np.arange(len(CHECKS)))
ax.set_yticklabels([f"{c} · {DECISIONS[c]}" for c in CHECKS])
ax.invert_yaxis()
ax.set_xlim(min(0.6, decisions["ci_low"].min() - 0.02), 1.005)
ax.set_xlabel("mean_score, mean over figures (95% bootstrap interval)")
ax.set_title("Layer 2, each check's decision")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=len(PRESENT), frameon=False)
fig.tight_layout()

## The comparison: E1, E2 and E3, pair by pair

Per figure, replicates averaged (counts pooled for E1 and E2):

| | endpoint |
|---|---|
| E1 | layer S, `correct / (correct + missing + spurious)`, rows pooled over the eleven checks |
| E2 | layer 1, correct applicability calls / all, pooled over the eleven checks |
| E3 | each check's decision, `mean_score` on applicable instances, one per check |

For each pair, the paired difference over the figures both answered, with a
95% percentile bootstrap interval: **better** if it excludes 0, **equivalent**
if it lies within ±0.02, **both** if both, **undecided** if neither.

In [ ]:
def per_figure_endpoints() -> pd.DataFrame:
    e1 = per_example_rate(rows_s.assign(all="all"), "outcome", ("correct_row",),
                          ("correct_row", "missing_row", "spurious_row"), by=("all",))
    e1 = e1.assign(endpoint="E1 layer S")
    e2 = per_example_rate(layer_1.assign(all="all"), "layer1", ("correct_applicable", "correct_NA"),
                          by=("all",)).assign(endpoint="E2 layer 1")
    d = scores.dropna(subset=["mean_score"])
    d = d[[is_decision(c, p) for c, p in zip(d["check"], d["property"])]]
    e3 = (d.groupby(["condition", "check", "example"])["mean_score"].mean().rename("rate").reset_index()
          .assign(endpoint=lambda f: "E3 " + f["check"]))
    keep = ["condition", "endpoint", "example", "rate"]
    return pd.concat([e1[keep], e2[keep], e3[keep]], ignore_index=True)


def classify(low: float, high: float) -> str:
    differs = low > 0 or high < 0
    within = low >= -MARGIN and high <= MARGIN
    return {(True, True): "both", (True, False): "better", (False, True): "equivalent",
            (False, False): "undecided"}[(differs, within)]


endpoints = per_figure_endpoints()
ENDPOINTS = ["E1 layer S", "E2 layer 1"] + [f"E3 {c}" for c in CHECKS]
rng = np.random.default_rng(SEED)
rows = []
for a, b in PAIRS:
    if a not in PRESENT or b not in PRESENT:
        continue
    for endpoint in ENDPOINTS:
        sub = endpoints[endpoints["endpoint"] == endpoint].pivot_table(
            index="example", columns="condition", values="rate")
        if a not in sub or b not in sub:
            continue
        diff = (sub[a] - sub[b]).dropna().to_numpy()
        if not len(diff):
            continue
        boots = rng.choice(diff, size=(RESAMPLES, len(diff)), replace=True).mean(axis=1)
        low, high = np.percentile(boots, [2.5, 97.5])
        verdict = classify(low, high)
        if verdict in ("better", "both"):
            verdict += f": {SHORT[a] if diff.mean() > 0 else SHORT[b]}"
        rows.append({"pair": f"{SHORT[a]} - {SHORT[b]}", "endpoint": endpoint,
                     "difference": diff.mean(), "ci_low": low, "ci_high": high,
                     "n_figures": len(diff), "verdict": verdict})
comparison = pd.DataFrame(rows)
comparison.pivot_table(index="endpoint", columns="pair", values="verdict", aggfunc="first").reindex(ENDPOINTS)

In [ ]:
comparison.round(4)

## The extension rule

After three replicates, the rule says extend — for all three models — if, for
any pair, E1 or E2 is undecided, or more than three of the eleven E3
decisions are. **It does not extend anything**: when it fires, the author is
asked, and the two further replicates run only on their go-ahead.

In [ ]:
rows = []
for pair, sub in comparison.groupby("pair"):
    undecided = sub[sub["verdict"] == "undecided"]["endpoint"].tolist()
    e12 = [e for e in undecided if not e.startswith("E3")]
    e3 = [e for e in undecided if e.startswith("E3")]
    rows.append({"pair": pair, "E1/E2 undecided": ", ".join(e12) or "-",
                 "E3 undecided": len(e3), "fires": bool(e12) or len(e3) > 3})
rule = pd.DataFrame(rows)
print("replicates:", scores.groupby("condition")["replicate"].nunique().to_dict())
print("THE RULE FIRES: ask the author before --extend" if rule["fires"].any() else "the rule does not fire")
rule

## Cost and time per figure

The CLI's cost estimates, at concurrency 50, the models one after another.
The background model's share is included in `cost` and shown apart. Token
counts are given so that cost can be recomputed at one price list.

In [ ]:
rng = np.random.default_rng(SEED)
per_figure = usage.groupby(["condition", "example"]).mean(numeric_only=True).reset_index()
rows = []
for cond, sub in per_figure.groupby("condition"):
    for q in ("cost", "background_cost", "input_tokens", "output_tokens", "turns", "seconds"):
        mean, low, high, n = bootstrap_mean(sub[q], rng)
        rows.append({"condition": cond, "quantity": q, "mean": mean, "ci_low": low,
                     "ci_high": high, "n_figures": n})
print("concurrency recorded:", usage["concurrency"].value_counts().to_dict())
print("total cost:", usage.groupby("condition")["cost"].sum().round(2).to_dict(),
      f"= ${usage['cost'].sum():.2f} over {len(usage)} sessions")
pd.DataFrame(rows).pivot_table(index="quantity", columns="condition", values="mean")[PRESENT]

## Sessions that did not invoke every check

Each beside the same figure's sessions of the same model that invoked all
eleven: layer S and layer 1 pooled over the checks, errors on the eleven
decisions counted (not averaged across properties), and size.

In [ ]:
KEY = ["condition", "replicate", "example"]


def session_rates() -> pd.DataFrame:
    s = (rows_s.groupby(KEY + ["outcome"])["count"].sum().unstack(fill_value=0))
    out = pd.DataFrame({
        "S_both": s["correct_row"] / s[["correct_row", "missing_row", "spurious_row"]].sum(axis=1),
        "missing_rows": s["missing_row"], "spurious_rows": s["spurious_row"],
    })
    l1 = layer_1.groupby(KEY + ["layer1"])["count"].sum().unstack(fill_value=0)
    out["layer_1"] = l1[["correct_applicable", "correct_NA"]].sum(axis=1) / l1.sum(axis=1)
    d = scores.dropna(subset=["mean_score"])
    d = d[[is_decision(c, p) for c, p in zip(d["check"], d["property"])]]
    d = d.assign(errors=(1 - d["mean_score"]) * d["n_scored"])
    out["decision_errors"] = d.groupby(KEY)["errors"].sum()
    out["decision_instances"] = d.groupby(KEY)["n_scored"].sum()
    out = out.join(usage.set_index(KEY)[["output_tokens", "turns", "cost"]])
    return out.join(sessions.set_index(KEY)[["n_checks", "dispatch"]]).reset_index()


per_session = session_rates()
columns = ["n_checks", "S_both", "missing_rows", "spurious_rows", "layer_1", "decision_errors",
           "decision_instances", "output_tokens", "turns", "cost"]
rows = []
for _, z in per_session[per_session["dispatch"] != "all eleven"].iterrows():
    same = per_session[(per_session["condition"] == z["condition"]) & (per_session["example"] == z["example"])
                       & (per_session["dispatch"] == "all eleven")]
    rows.append({"condition": z["condition"], "example": z["example"], "session": z["dispatch"],
                 "replicate": z["replicate"], **z[columns].to_dict()})
    rows.append({"condition": z["condition"], "example": z["example"],
                 "session": f"all eleven (mean of {len(same)})", "replicate": None,
                 **(same[columns].mean().to_dict() if len(same) else {})})
short_dispatch = pd.DataFrame(rows)
short_dispatch.set_index(["condition", "example", "session"]).round(3) if rows else "every session invoked all eleven"

## Error analysis: the candidates, on the decisions

Every layer-1 applicability error and layer-2 disagreement on the eleven
decisions, per model, and those recurring in at least two of the three
replicates of a model, for the manual sorting — **model error**, **gold
error** or **ambiguous**. Written to
`experiments/exp-07-models/error_analysis_candidates.csv` (not committed) when
run on the full run; annotate a copy.

In [ ]:
def row_index(path: str) -> int | None:
    head = path.split("].", 1)[0]
    return int(head.rsplit("[", 1)[1]) if "[" in head else None


rows = []
for model, replicate, leaf in model_leaves():
    analysis = json.loads((leaf / ANALYSIS_FILENAME).read_text(encoding="utf-8"))
    for record in analysis.get("flat", []):
        example = (record.get("metadata") or {}).get("source")
        gold = record.get("expected_output") or {}
        for inst in (record.get("analysis") or {}).get("instances", []):
            check = owning_check(inst["leaf_property"])
            prop = "outputs" + inst["leaf_property"][len(check):]
            if not is_decision(check, prop):
                continue
            l1, l2, score = inst.get("layer1"), inst.get("layer2"), inst.get("score")
            list_error = l1 is None and l2 is None and score is not None and score < 1
            if l1 not in LAYER1_OUTLIER_OUTCOMES and l2 not in LAYER2_ERROR_OUTCOMES and not list_error:
                continue
            i = row_index(inst["path"])
            gold_rows = gold.get(check) or []
            panel = gold_rows[i].get("panel_label") if i is not None and i < len(gold_rows) else None
            rows.append({
                "condition": model, "replicate": replicate, "example": example, "check": check,
                "panel": panel, "property": prop,
                "outcome": l1 if l1 in LAYER1_OUTLIER_OUTCOMES else l2 if l2 in LAYER2_ERROR_OUTCOMES
                else f"list score {score:.2f}",
                "exp_value": json.dumps(inst.get("exp_value"), ensure_ascii=False),
                "pred_value": json.dumps(inst.get("pred_value"), ensure_ascii=False),
            })
errors = pd.DataFrame(rows)
print(f"{len(errors)} decision error instances")
(errors.groupby(["check", "condition"]).size().unstack(fill_value=0).reindex(CHECKS)
 .reindex(columns=PRESENT).fillna(0).astype(int)) if rows else errors

In [ ]:
recurring = (errors.groupby(["condition", "example", "check", "panel", "property", "exp_value"])
             .agg(sessions_wrong=("replicate", "nunique"),
                  most_common_pred=("pred_value", lambda s: Counter(s).most_common(1)[0][0]),
                  outcome=("outcome", lambda s: Counter(s).most_common(1)[0][0]))
             .reset_index()
             .sort_values(["sessions_wrong", "check"], ascending=[False, True])) if len(errors) else errors
# Wrong in the same way across models is a gold candidate before it is a model one.
if len(recurring):
    models_wrong = recurring.groupby(["example", "check", "panel", "property"])["condition"].nunique()
    recurring["models_wrong"] = [models_wrong[k] for k in
                                 zip(recurring["example"], recurring["check"], recurring["panel"], recurring["property"])]
    recurring["sorting"], recurring["note"] = "", ""
    out = Path("../../experiments/exp-07-models/error_analysis_candidates.csv")
    picked = recurring[recurring["sessions_wrong"] >= 2]
    if "EXP07_ROOT" in os.environ:
        print(f"{len(picked)} recurring; not written -- EXP07_ROOT points away from the full run")
    else:
        picked.to_csv(out, index=False)
        print(f"{len(picked)} decision disagreements wrong in at least 2 replicates, written to {out.name}")
recurring.head(40)

## What this does not do

It does not extend the run — the author decides — and it does not sort the
disagreements: that is the manual error analysis, recorded in
[`thinking/experiments/exp-07-models.md`](../../thinking/experiments/exp-07-models.md).